<a href="https://colab.research.google.com/github/Atharv2200/SpendDna/blob/main/SpendDna_Atharv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('rahul_transactions.csv')
initial_rows = len(df)

df = df.drop_duplicates()
dropped_count = initial_rows - len(df)

df['date'] = pd.to_datetime(df['Date'], errors='coerce', dayfirst=True)

df['Amount_str'] = df['Amount'].astype(str)
df['Amount_str'] = df['Amount_str'].str.replace('Rs.', '', regex=False)
df['Amount_str'] = df['Amount_str'].str.replace(',', '', regex=False)
df['Amount_str'] = df['Amount_str'].str.replace(' ', '', regex=False)
df['amount'] = pd.to_numeric(df['Amount_str'], errors='coerce')

df['Type'] = df['Type'].astype(str).str.lower().str.strip()
df.loc[df['Type'] == 'dr', 'Type'] = 'debit'
df.loc[df['Type'] == 'cr', 'Type'] = 'credit'

# --- THE BUG FIX ---
# Drops the rows that couldn't be parsed into proper dates or numbers
df = df.dropna(subset=['date', 'amount'])

print(f"Parsed {len(df)} transactions across 6 months.")
print(f"Dropped {dropped_count} duplicates.")
print(f"{df['date'].isna().sum()} unparseable dates, {df['amount'].isna().sum()} unparseable amounts.")

In [ ]:
vendor_keywords = {
    'Swiggy': ['SWIGGY', 'BUNDL'],
    'Zomato': ['ZOMATO'],
    'Zepto': ['ZEPTO', 'BLINKIT', 'INSTAMART'],
    'Amazon': ['AMAZON', 'AMZN'],
    'Myntra': ['MYNTRA'],
    'Zerodha': ['ZERODHA'],
    'BookMyShow': ['BOOKMYSHOW', 'BMS'],
    'Uber': ['UBER'],
    'Ola': ['OLA'],
    'Starbucks': ['STARBUCKS', 'TATA STARBUCKS'],
    'Third Wave': ['THIRD WAVE', 'TWCR'],
    'P2P Transfer': ['UPI-ANKIT', 'UPI-PRIYA', 'UPI-RAHUL'],
    'Cash Withdrawal': ['ATM-WDL']
}

def extract_vendor_name(description):
    desc = str(description).lower()
    for vendor, keywords in vendor_keywords.items():
        for word in keywords:
            if word.lower() in desc:
                return vendor
    return 'Uncategorised'

df['vendor_clean'] = df['Description'].apply(extract_vendor_name)

print(f"Found {df['vendor_clean'].nunique()} unique vendors.")
print("\nTop 5 Vendors by transaction count:")
print(df['vendor_clean'].value_counts().head(5))

In [ ]:
category_mapping = {
    'Swiggy': 'Food Delivery',
    'Zomato': 'Food Delivery',
    'Zepto': 'Quick Commerce',
    'Amazon': 'E-commerce',
    'Myntra': 'E-commerce',
    'Zerodha': 'Investments',
    'BookMyShow': 'Entertainment',
    'Uber': 'Transport',
    'Ola': 'Transport',
    'Starbucks': 'Cafe',
    'Third Wave': 'Cafe',
    'P2P Transfer': 'Personal Transfer',
    'Cash Withdrawal': 'Cash Withdrawal'
}

df['category'] = df['vendor_clean'].map(category_mapping).fillna('Uncategorised')

print("Transaction count by category:")
print(df['category'].value_counts())

In [ ]:
debits = df[df['Type'] == 'debit']
credits = df[df['Type'] == 'credit']

total_credits = credits['amount'].sum()
total_debits = debits['amount'].sum()
savings_rate = ((total_credits - total_debits) / total_credits) * 100

debits['month'] = debits['date'].dt.month
monthly_trend = debits.pivot_table(values='amount', index='category', columns='month', aggfunc='sum').fillna(0)

print(f"Total Credits: Rs. {total_credits:,.0f}")
print(f"Total Debits: Rs. {total_debits:,.0f}")
print(f"Savings Rate: {savings_rate:.1f}%")
print("\nMonthly Spending Trend (Sample):")
print(monthly_trend.head(3))

In [ ]:
debits['hour'] = debits['Time'].astype(str).str[:2].astype(int)

food_delivery = debits[debits['category'] == 'Food Delivery']
late_night_orders = food_delivery[(food_delivery['hour'] >= 21) | (food_delivery['hour'] <= 2)]

if len(food_delivery) > 0:
    late_night_pct = (len(late_night_orders) / len(food_delivery)) * 100
    print(f"{late_night_pct:.1f}% of Food Delivery happens between 9 PM and 2 AM.")

In [ ]:
cat_means = debits.groupby('category')['amount'].transform('mean')
cat_stds = debits.groupby('category')['amount'].transform('std')

debits['z_score'] = (debits['amount'] - cat_means) / cat_stds
anomalies = debits[debits['z_score'] > 2].sort_values('z_score', ascending=False)

print(f"Flagged {len(anomalies)} anomalous transactions.")
print("\nTop 3 Anomalies:")
print(anomalies[['date', 'vendor_clean', 'category', 'amount', 'z_score']].head(3))

In [ ]:
archetypes = []
category_totals = debits.groupby('category')['amount'].sum() / total_debits

food_spend = category_totals.get('Food Delivery', 0) + category_totals.get('Restaurants', 0) + category_totals.get('Cafe', 0)
if food_spend > 0.25:
    archetypes.append(f"-> THE FOODIE ({food_spend*100:.1f}% on food)")

if category_totals.get('Quick Commerce', 0) > 0.15:
    archetypes.append(f"-> THE QUICK COMMERCE JUNKIE ({category_totals['Quick Commerce']*100:.1f}% on Q-com)")

if category_totals.get('E-commerce', 0) > 0.15:
    archetypes.append(f"-> THE SHOPAHOLIC ({category_totals['E-commerce']*100:.1f}% on e-commerce)")

if category_totals.get('Investments', 0) > 0.15:
    archetypes.append(f"-> THE INVESTOR ({category_totals['Investments']*100:.1f}% on SIPs)")

if len(food_delivery) > 0 and (len(late_night_orders) / len(food_delivery)) > 0.5:
    archetypes.append(f"-> THE LATE-NIGHT SNACKER ({(len(late_night_orders) / len(food_delivery))*100:.0f}% food after 9 PM)")

if savings_rate < 10:
    archetypes.append(f"-> THE YOLO SPENDER (savings rate {savings_rate:.1f}%)")

print("Archetypes Detected:")
for a in archetypes:
    print(a)

In [ ]:
print("=" * 66)
print("SpendDNA REPORT")
print("RAHUL SHARMA | Jan to Jun 2024")
print(f"6 months | {len(df)} transactions")
print("=" * 66)

print("\nEXECUTIVE SUMMARY")
print(f"{'Total credits':<15}: Rs. {total_credits:,.0f}")
print(f"{'Total debits':<15}: Rs. {total_debits:,.0f}")
print(f"{'Net change':<15}: Rs. {total_credits - total_debits:,.0f} (overspending)")
print(f"{'Savings rate':<15}: {savings_rate:.1f}%")
print(f"{'Transactions':<15}: {len(df)}")
print(f"{'Unique vendors':<15}: {df['vendor_clean'].nunique()}")

print("\nTOP CATEGORIES (% of debit total)")
cat_spend_sorted = debits.groupby('category')['amount'].sum().sort_values(ascending=False)
for cat, val in cat_spend_sorted.head(5).items():
    pct = (val / total_debits) * 100
    bars = "#" * int(pct / 1.5)
    print(f"{cat:<15} {bars:<20} {pct:>5.1f}%   Rs. {val:,.0f}")

print("\nTOP ANOMALIES (3+ stddev from category mean)")
for _, row in anomalies.head(3).iterrows():
    date_str = row['date'].strftime('%d %b')
    print(f"{date_str:<10} {row['vendor_clean']:<15} Rs. {row['amount']:>8,.0f} (z={row['z_score']:.1f})")

print("\nRAHUL'S SPENDING ARCHETYPES")
for arc in archetypes:
    print(arc)

print("\n" + "=" * 66)